<a href="https://colab.research.google.com/github/Arjin-Yoganantham/DAA---LAB-EXPERIMENTS/blob/main/exp1b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import statistics
import time
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

# ── Floating-point tolerance for "found" ─────────────────────
EPSILON = 1e-9

BLUE   = "#00d4ff"
ORANGE = "#ff6b35"
GREEN  = "#00e676"
PURPLE = "#c792ea"
BG     = "#0d0f14"
PANEL  = "#131720"
GRID_C = "#1e2535"
TEXT   = "#e8ecf4"
MUTED  = "#5a6478"
SIZES  = [10_000, 50_000, 100_000]
SIZE_LABELS = ["10,000", "50,000", "100,000"]
SAMPLES_PER_SIZE = 200   # targets tested per dataset size


# ─────────────────────────────────────────────────────────────
#  ALGORITHM 1 — INTERPOLATION SEARCH  (float-aware)
# ─────────────────────────────────────────────────────────────

def interpolation_search_float(arr, target, trace=False):
    """
    Interpolation Search adapted for sorted floating-point arrays.

    Changes vs integer version
    ──────────────────────────
    1. Position uses  int(round(...))  not floor division (//).
    2. Match uses  abs(arr[pos] - target) < EPSILON  not ==.
    3. Loop guard checks arr[hi] != arr[lo] to avoid ZeroDivision
       (all values equal → not possible in a uniform distribution,
        but defensive coding is good practice).

    Complexity on uniform floats
    ─────────────────────────────
    Average : O(log log n)
    Worst   : O(n)
    """
    lo, hi = 0, len(arr) - 1
    comparisons = 0

    while lo <= hi:
        comparisons += 1  # range check counts as comparison
        if arr[lo] > target or arr[hi] < target:
            break

        # ── ZeroDivision guard ───────────────────────────────
        if abs(arr[hi] - arr[lo]) < EPSILON:
            if abs(arr[lo] - target) < EPSILON:
                return lo, comparisons
            break

        # ── Float interpolation formula ──────────────────────
        ratio = (target - arr[lo]) / (arr[hi] - arr[lo])
        pos   = lo + int(round(ratio * (hi - lo)))
        pos   = max(lo, min(hi, pos))   # clamp to valid range

        comparisons += 1
        if abs(arr[pos] - target) < EPSILON:
            return pos, comparisons
        elif arr[pos] < target:
            lo = pos + 1
        else:
            hi = pos - 1

    return -1, comparisons


# ─────────────────────────────────────────────────────────────
#  ALGORITHM 2 — BINARY SEARCH  (float-aware)
# ─────────────────────────────────────────────────────────────

def binary_search_float(arr, target):
    """
    Binary Search adapted for floating-point arrays.
    Uses EPSILON tolerance for equality check.
    Complexity: O(log n) regardless of distribution.
    """
    lo, hi = 0, len(arr) - 1
    comparisons = 0

    while lo <= hi:
        mid = (lo + hi) // 2
        comparisons += 1
        if abs(arr[mid] - target) < EPSILON:
            return mid, comparisons
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1

    return -1, comparisons


# ─────────────────────────────────────────────────────────────
#  DATASET GENERATOR
# ─────────────────────────────────────────────────────────────

def generate_dataset(n, lo=0.0, hi=1000.0, seed=None):
    """
    Generate n unique floats uniformly distributed in [lo, hi],
    returned as a sorted list.
    """
    rng = random.Random(seed)
    data = sorted(rng.uniform(lo, hi) for _ in range(n))
    return data


# ─────────────────────────────────────────────────────────────
#  ANALYSIS ENGINE
# ─────────────────────────────────────────────────────────────

def analyze_size(n, n_samples=SAMPLES_PER_SIZE):
    """
    Build a dataset of size n, pick n_samples targets from it,
    run both algorithms, return per-query comparison counts.
    """
    arr = generate_dataset(n, seed=42 + n)

    # Pick targets that are guaranteed to exist in the array
    target_indices = random.sample(range(n), min(n_samples, n))
    targets = [arr[i] for i in target_indices]

    i_comps, b_comps = [], []
    i_times, b_times = [], []

    for t in targets:
        t0 = time.perf_counter()
        _, ic = interpolation_search_float(arr, t)
        i_times.append(time.perf_counter() - t0)
        i_comps.append(ic)

        t0 = time.perf_counter()
        _, bc = binary_search_float(arr, t)
        b_times.append(time.perf_counter() - t0)
        b_comps.append(bc)

    return {
        "n":        n,
        "i_comps":  i_comps,
        "b_comps":  b_comps,
        "i_avg":    statistics.mean(i_comps),
        "b_avg":    statistics.mean(b_comps),
        "i_med":    statistics.median(i_comps),
        "b_med":    statistics.median(b_comps),
        "i_max":    max(i_comps),
        "b_max":    max(b_comps),
        "i_min":    min(i_comps),
        "b_min":    min(b_comps),
        "i_std":    statistics.stdev(i_comps),
        "b_std":    statistics.stdev(b_comps),
        "i_ms":     statistics.mean(i_times) * 1000,
        "b_ms":     statistics.mean(b_times) * 1000,
    }


# ─────────────────────────────────────────────────────────────
#  CONSOLE REPORT
# ─────────────────────────────────────────────────────────────

def print_report(results):
    W = 70
    print(f"\n{'═' * W}")
    print(f"  FLOATING-POINT INTERPOLATION vs BINARY SEARCH")
    print(f"  Range: 0.0 – 1000.0  |  {SAMPLES_PER_SIZE} random targets per size")
    print(f"{'═' * W}")
    hdr = f"  {'Dataset':>10}  {'Algorithm':<22} {'Avg':>6} {'Med':>6} {'Min':>5} {'Max':>5} {'Std':>6}  {'Avg(ms)':>8}"
    print(hdr)
    print(f"{'─' * W}")

    for r in results:
        label = f"{r['n']:,}"
        theory_i = math.log2(math.log2(r['n'])) if r['n'] > 2 else 1
        theory_b = math.log2(r['n'])
        print(f"  {label:>10}  {'Interpolation':<22} {r['i_avg']:>6.2f} {r['i_med']:>6.1f} "
              f"{r['i_min']:>5} {r['i_max']:>5} {r['i_std']:>6.2f}  {r['i_ms']:>8.4f}")
        print(f"  {'':>10}  {'Binary Search':<22} {r['b_avg']:>6.2f} {r['b_med']:>6.1f} "
              f"{r['b_min']:>5} {r['b_max']:>5} {r['b_std']:>6.2f}  {r['b_ms']:>8.4f}")
        print(f"  {'':>10}  {'Theory log₂(log₂n)':<22} {theory_i:>6.2f}  "
              f"{'Theory log₂n':<8} {theory_b:>6.2f}")
        print(f"{'─' * W}")

    print()

    # Single demo trace
    print(f"\n{'─' * W}")
    print(f"  SINGLE SEARCH DEMO  (n = 10,000 dataset)")
    print(f"{'─' * W}")
    demo_arr = generate_dataset(10_000, seed=99)
    demo_target = demo_arr[random.randint(100, 9900)]
    print(f"  Target value : {demo_target:.6f}")

    _, ic = interpolation_search_float(demo_arr, demo_target)
    _, bc = binary_search_float(demo_arr, demo_target)
    print(f"  Interpolation comparisons : {ic}")
    print(f"  Binary Search comparisons : {bc}")
    winner = "Interpolation" if ic < bc else "Binary Search" if bc < ic else "TIE"
    print(f"  ⚡ Fewer comparisons     : {winner}")
    print(f"{'─' * W}\n")


# ─────────────────────────────────────────────────────────────
#  PLOTTING
# ─────────────────────────────────────────────────────────────

def style_ax(ax, title=""):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_C)
    ax.grid(color=GRID_C, linewidth=0.7, linestyle="--", zorder=0)
    if title:
        ax.set_title(title, color=TEXT, fontsize=10, fontweight="bold", pad=10)


def plot_results(results):
    fig = plt.figure(figsize=(18, 13), facecolor=BG)
    fig.suptitle(
        "Interpolation Search on Floating-Point Numbers  (0.0 – 1000.0)\n"
        "Comparison Count Analysis · Dataset Sizes: 10,000 · 50,000 · 100,000",
        color=TEXT, fontsize=13, fontweight="bold", y=0.98
    )

    gs = GridSpec(2, 3, figure=fig, hspace=0.44, wspace=0.36,
                  left=0.07, right=0.97, top=0.90, bottom=0.08)

    ns     = [r["n"]     for r in results]
    i_avgs = [r["i_avg"] for r in results]
    b_avgs = [r["b_avg"] for r in results]
    i_maxs = [r["i_max"] for r in results]
    b_maxs = [r["b_max"] for r in results]
    i_stds = [r["i_std"] for r in results]
    b_stds = [r["b_std"] for r in results]

    # Theory curves
    theory_ns = [1000, 5000, 10000, 30000, 50000, 75000, 100000]
    t_interp  = [math.log2(math.log2(n)) for n in theory_ns]
    t_binary  = [math.log2(n)            for n in theory_ns]

    x = range(len(SIZES))
    bar_w = 0.35
    x_i = [xi - bar_w / 2 for xi in x]
    x_b = [xi + bar_w / 2 for xi in x]

    # ── 1. Grouped bar — average comparisons ─────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    style_ax(ax1, "Avg Comparisons per Dataset Size")
    bars_i = ax1.bar(x_i, i_avgs, bar_w, color=BLUE,   label="Interpolation", zorder=3)
    bars_b = ax1.bar(x_b, b_avgs, bar_w, color=ORANGE, label="Binary Search",  zorder=3)
    for bar, val in zip(list(bars_i) + list(bars_b), i_avgs + b_avgs):
        ax1.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.15,
                 f"{val:.1f}", ha="center", va="bottom",
                 color=TEXT, fontsize=8, fontweight="bold")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(SIZE_LABELS, color=MUTED)
    ax1.set_ylabel("Avg Comparisons", color=MUTED)
    ax1.set_xlabel("Dataset Size (n)", color=MUTED)
    ax1.legend(facecolor=PANEL, edgecolor=GRID_C, labelcolor=TEXT, fontsize=8)

    # ── 2. Line — avg vs theory ───────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    style_ax(ax2, "Avg Comparisons vs Theoretical Complexity")
    ax2.plot(theory_ns, t_interp, color=BLUE,   linewidth=1.5,
             linestyle="--", label="O(log log n) theory")
    ax2.plot(theory_ns, t_binary, color=ORANGE, linewidth=1.5,
             linestyle="--", label="O(log n) theory")
    ax2.plot(ns, i_avgs, color=BLUE,   linewidth=2, marker="o",
             markersize=8, label="Interpolation actual")
    ax2.plot(ns, b_avgs, color=ORANGE, linewidth=2, marker="s",
             markersize=8, label="Binary actual")
    ax2.set_xlabel("Dataset Size (n)", color=MUTED)
    ax2.set_ylabel("Comparisons", color=MUTED)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax2.legend(facecolor=PANEL, edgecolor=GRID_C, labelcolor=TEXT, fontsize=7.5)

    # ── 3. Max comparisons bar ────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    style_ax(ax3, "Max (Worst-Case) Comparisons")
    bars_im = ax3.bar(x_i, i_maxs, bar_w, color=BLUE,   alpha=0.85, zorder=3)
    bars_bm = ax3.bar(x_b, b_maxs, bar_w, color=ORANGE, alpha=0.85, zorder=3)
    for bar, val in zip(list(bars_im) + list(bars_bm), i_maxs + b_maxs):
        ax3.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.1,
                 str(val), ha="center", va="bottom",
                 color=TEXT, fontsize=8, fontweight="bold")
    ax3.set_xticks(list(x))
    ax3.set_xticklabels(SIZE_LABELS, color=MUTED)
    ax3.set_ylabel("Max Comparisons", color=MUTED)
    ax3.set_xlabel("Dataset Size (n)", color=MUTED)
    ax3.legend(handles=[
        mpatches.Patch(color=BLUE,   label="Interpolation"),
        mpatches.Patch(color=ORANGE, label="Binary Search"),
    ], facecolor=PANEL, edgecolor=GRID_C, labelcolor=TEXT, fontsize=8)

    # ── 4,5,6. Per-size violin/distribution plots ─────────────
    for col, r in enumerate(results):
        ax = fig.add_subplot(gs[1, col])
        style_ax(ax, f"Comparison Distribution  (n = {r['n']:,})")

        vp = ax.violinplot(
            [r["i_comps"], r["b_comps"]],
            positions=[1, 2],
            showmedians=True,
            showextrema=True,
        )
        colors_v = [BLUE, ORANGE]
        for i, body in enumerate(vp["bodies"]):
            body.set_facecolor(colors_v[i])
            body.set_edgecolor(colors_v[i])
            body.set_alpha(0.5)
        for part in ["cmedians", "cmins", "cmaxes", "cbars"]:
            vp[part].set_color(TEXT)
            vp[part].set_linewidth(1.2)

        # Overlay scatter jitter
        for idx, (data, col_hex) in enumerate(
                [(r["i_comps"], BLUE), (r["b_comps"], ORANGE)], 1):
            jitter = [idx + random.uniform(-0.08, 0.08) for _ in data]
            ax.scatter(jitter, data, color=col_hex, alpha=0.25,
                       s=6, zorder=3)

        # Annotation: avg lines
        ax.axhline(r["i_avg"], color=BLUE,   linewidth=1.2,
                   linestyle=":", alpha=0.9)
        ax.axhline(r["b_avg"], color=ORANGE, linewidth=1.2,
                   linestyle=":", alpha=0.9)
        ax.text(0.5, r["i_avg"] + 0.05, f"avg {r['i_avg']:.1f}",
                color=BLUE,   fontsize=7, transform=ax.get_yaxis_transform())
        ax.text(0.5, r["b_avg"] + 0.05, f"avg {r['b_avg']:.1f}",
                color=ORANGE, fontsize=7, transform=ax.get_yaxis_transform())

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Interpolation", "Binary"], color=MUTED, fontsize=8)
        ax.set_ylabel("Comparisons", color=MUTED)

    # Legend
    p1 = mpatches.Patch(color=BLUE,   label="Interpolation Search")
    p2 = mpatches.Patch(color=ORANGE, label="Binary Search")
    fig.legend(handles=[p1, p2], loc="lower center", ncol=2,
               facecolor=PANEL, edgecolor=GRID_C, labelcolor=TEXT,
               fontsize=9, bbox_to_anchor=(0.5, 0.01))

    plt.savefig("float_search_analysis.png", dpi=150,
                bbox_inches="tight", facecolor=BG)
    print("  Chart saved → float_search_analysis.png")
    plt.show()


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    random.seed(0)

    print("\n  Running analysis across dataset sizes...")
    results = []
    for n in SIZES:
        print(f"    n = {n:>7,}  ...", end="", flush=True)
        r = analyze_size(n)
        results.append(r)
        print(f"  done  (interp avg={r['i_avg']:.2f}, binary avg={r['b_avg']:.2f})")

    print_report(results)
    plot_results(results)


if __name__ == "__main__":
    main()